# Lesson 2 — The Robot Knows Where It Is

*In Lesson 1 you ran `rqt_graph` and saw a topic called `/tf`. Every arrow, every node — you understood them. Except that one. Today we open it.*

---

## 🎬 Watch first — then do

<div style="padding:56.25% 0 0 0;position:relative;">
  <iframe src="https://player.vimeo.com/video/YOUR_VIDEO_ID"
          style="position:absolute;top:0;left:0;width:100%;height:100%;"
          frameborder="0" allow="autoplay; fullscreen; picture-in-picture" allowfullscreen>
  </iframe>
</div>

---

## 🚀 Start the robot

Open Terminator and run:

```bash
bash ~/course_materials/scripts/start_robot.sh
```

RViz opens automatically. Your robot appears — the same one from Lesson 1.

---

## 🔍 Open the mysterious topic

In a second terminal, run:

```bash
ros2 topic echo /tf
```

You will see a continuous stream of data — parent frames, child frames, translations, rotations, timestamps.

This is the robot continuously publishing where every part of its body is in space.

Press **Ctrl+C** to stop.

---

## 🎯 What Are TFs?

A **TF (Transform)** represents the spatial relationship between two coordinate frames in 3D space.

TFs allow ROS2 to track how different parts of a robot are positioned and how they move over time.

![Links and frames](images/page_1.png)

---

## 🔗 Links, Frames, and TFs

- **Link**: one rigid part of the robot — chassis, wheel, caster
- **Frame**: the coordinate system attached to that link
- **TF**: the relationship between two frames — position and rotation

Your robot has these links: `base_footprint`, `base_link`, `base_left_wheel_link`, `base_right_wheel_link`, `caster_wheel_link`.

In RViz, open **RobotModel → Links** in the left panel to show or hide individual parts.

![Links and frames](images/page_2.png)

---

## 🧭 Coordinate Systems — the Right-Hand Rule

ROS2 uses the **right-hand rule** for all coordinate frames:

<div style="display:flex;align-items:center;gap:32px;">
<div>

- **X** (red): forward
- **Y** (green): left
- **Z** (blue): up

Every link in the robot has its own local X, Y, Z axes. When you see colored arrows in RViz, you are seeing these axes.

</div>
<img src="images/hand_right.png" style="height:180px;"/>
</div>

![Frames in RViz](images/page_3.png)

---

## 🌳 The TF Tree — parent and child frames

TFs are organized as a **tree**:
- Every frame has exactly one parent
- A frame can have many children
- Moving a parent moves all its children

Your robot's TF tree:
```
base_footprint
└── base_link
    ├── base_left_wheel_link
    ├── base_right_wheel_link
    └── caster_wheel_link
```

`base_footprint` is fixed to the ground. `base_link` is the chassis. The wheels move relative to `base_link`.

![TF tree structure](images/page_4.png)

---

## 📡 The /tf topic

Run in the terminal:

```bash
ros2 topic echo /tf
```

Each message contains:
- Parent frame and child frame
- Translation (x, y, z)
- Rotation (as a quaternion)
- Timestamp

Drive the robot and watch the values change in real time.

![/tf topic](images/page_5.png)

---

## 🗺️ Visualizing the TF tree

In the terminal:

```bash
cd /tmp && ros2 run tf2_tools view_frames
evince /tmp/frames_*.pdf
```

This generates a diagram showing every frame and how they connect.

![TF tree diagram](images/page_6.png)

---

## ❓ Why Do We Need TFs?

Every robotics system must always know:
- Where each part is located
- How parts move relative to each other
- How sensor data relates to the robot body

TFs solve this problem in a unified, scalable way. Every algorithm in ROS2 — navigation, manipulation, perception — reads from the TF tree.

![Why TFs matter](images/page_7.png)

---

## 🔧 Widget — Inspect your robot's URDF

Your robot is described by a URDF file generated when `start_robot.sh` runs. Run the cell below to parse it and display the full kinematic structure.

In [6]:
import xml.etree.ElementTree as ET
from IPython.display import display, HTML

urdf_path = '/tmp/robot.urdf'
root = None

try:
    tree = ET.parse(urdf_path)
    root = tree.getroot()
except FileNotFoundError:
    display(HTML('<div style="color:#FF4444;font-family:monospace;padding:12px;background:#1a0000;border-radius:8px;">'                 '⚠️ File not found. Make sure start_robot.sh is running.</div>'))

if root is not None:
    links = {}
    for link in root.findall('link'):
        name = link.get('name')
        geom_type, dims = 'virtual', ''
        visual = link.find('visual')
        if visual is not None:
            geom = visual.find('geometry')
            if geom is not None:
                for child in geom:
                    geom_type = child.tag
                    if child.tag == 'cylinder':
                        dims = f"r={child.get('radius')}m  l={child.get('length')}m"
                    elif child.tag == 'box':
                        dims = f"size={child.get('size')}"
                    elif child.tag == 'sphere':
                        dims = f"r={child.get('radius')}m"
                    elif child.tag == 'mesh':
                        dims = child.get('filename', '').split('/')[-1]
        links[name] = {'geom_type': geom_type, 'dims': dims}

    joints = []
    for joint in root.findall('joint'):
        parent = joint.find('parent').get('link')
        child  = joint.find('child').get('link')
        jtype  = joint.get('type')
        origin = joint.find('origin')
        xyz = origin.get('xyz', '0 0 0') if origin is not None else '0 0 0'
        joints.append({'name': joint.get('name'), 'type': jtype,
                       'parent': parent, 'child': child, 'xyz': xyz})

    children_map = {}
    for j in joints:
        children_map.setdefault(j['parent'], []).append(j)
    root_link = next(n for n in links if n not in {j['child'] for j in joints})

    jtype_color = {'fixed': '#555', 'continuous': '#00D4FF', 'revolute': '#FFA500', 'prismatic': '#00FF88'}
    jtype_icon  = {'fixed': '🔒', 'continuous': '🔄', 'revolute': '↺', 'prismatic': '↔️'}

    def render(link_name, depth=0, joint=None):
        indent = '&nbsp;' * (depth * 6)
        connector = '└── ' if depth > 0 else ''
        info = links.get(link_name, {})
        badge, xyz_info = '', ''
        if joint:
            color = jtype_color.get(joint['type'], '#888')
            icon  = jtype_icon.get(joint['type'], '')
            badge = f'<span style="background:{color}22;border:1px solid {color};color:{color};border-radius:4px;padding:1px 6px;font-size:11px;margin-right:6px;">{icon} {joint["type"]}</span>'
            xyz_info = f'<span style="color:#333;font-size:11px;margin-left:8px;">@ {joint["xyz"]}</span>'
        html = (f'<div style="margin:3px 0;">{indent}'
                f'<span style="color:#888;">{connector}</span>{badge}'
                f'<span style="color:#fff;font-weight:700;">{link_name}</span> '
                f'<span style="color:#555;font-size:12px;">{info["geom_type"]}</span> '
                f'<span style="color:#00D4FF;font-size:12px;">{info["dims"]}</span>'
                f'{xyz_info}</div>')
        for child_joint in children_map.get(link_name, []):
            html += render(child_joint['child'], depth + 1, child_joint)
        return html

    legend = ''.join([
        f'<span style="background:{color}22;border:1px solid {color};color:{color};border-radius:4px;padding:2px 8px;font-size:12px;margin-right:8px;">'
        f'{jtype_icon.get(jt,"")} {jt}</span>'
        for jt, color in jtype_color.items()
    ])

    display(HTML(f'''
<div style="font-family:monospace;background:#0a0a0a;border:1px solid #333;border-radius:8px;padding:24px;max-width:750px;">
  <div style="color:#aaa;font-size:13px;margin-bottom:4px;">Robot: <span style="color:#fff;">{root.get("name", "unknown")}</span></div>
  <div style="color:#aaa;font-size:13px;margin-bottom:16px;">Links: <span style="color:#fff;">{len(links)}</span> &nbsp;&nbsp; Joints: <span style="color:#fff;">{len(joints)}</span></div>
  <div style="margin-bottom:16px;">{legend}</div>
  <div style="border-top:1px solid #222;padding-top:16px;">{render(root_link)}</div>
</div>
'''))

---

## 🧠 Challenge — Explore your robot's tree

Look at the widget output above and answer:

1. What is the **root link** — the one at the top with no parent?
2. How many **continuous** joints does your robot have? What parts are they?
3. Drive the robot in noVNC while watching `ros2 topic echo /tf`. Which values change as the robot moves?

*Write your answers here:*

**1. Root link:**

**2. Continuous joints:**

**3. Values that change when driving:**

---

## ✅ What you learned

- A **link** is a rigid part of the robot — chassis, wheel, caster
- A **frame** is the coordinate system attached to each link
- A **TF** is the relationship between two frames — position and rotation
- TFs are organized as a **tree** — moving a parent moves all its children
- The `/tf` topic carries these relationships in real time
- `view_frames` generates a diagram of the full tree

---

### 📋 Commands reference

| Command | What it does |
|---|---|
| `ros2 topic echo /tf` | See coordinate frame transforms in real time |
| `ros2 run tf2_tools view_frames` | Generate a diagram of the full TF tree |

---

> You now know what the robot is made of and how its parts relate in space.
>
> But you've only *read* a robot — you haven't *built* one. In the next lesson you will write a URDF from scratch: link by link, joint by joint.
>
> **Next lesson: build your own robot.**